In [24]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="/Users/solomon/mnist-cnn/data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="/Users/solomon/mnist-cnn/data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [25]:
#hyperparameters
#these are "adjustable parameters that let you control the model optimization process"
learning_rate = 1e-3
batch_size = 64
epochs = 10


In [26]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()
#this combines log softmax with negative log likelihood loss

In [27]:
#SGD (stochastic gradient descent) takes data samples individually, or in this case in batches of 64 and computes the loss gradient over those batches to minimize it
#the path is noisy which promotes generalization and reduces the chance of overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [28]:
#train and test loop

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #runs forward prop through the model
        pred = model(X)
        #computes the cross entropy loss of the models prediction vs the label
        loss = loss_fn(pred, y)

        #running back prop
        loss.backward()
        optimizer.step()
        #clears grad and frees memory
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            #weird regex lol, copied from pytorch docs
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    #intentionally disabling the gradient to speed up computation
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [29]:
#running the loop

loss_fn = nn.CrossEntropyLoss()

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.305332  [   64/60000]
loss: 0.543887  [ 6464/60000]
loss: 0.376429  [12864/60000]
loss: 0.496569  [19264/60000]
loss: 0.450015  [25664/60000]
loss: 0.425838  [32064/60000]
loss: 0.362874  [38464/60000]
loss: 0.541733  [44864/60000]
loss: 0.490855  [51264/60000]
loss: 0.534219  [57664/60000]
Test Error: 
 Accuracy: 84.3%, Avg loss: 0.429306 

Epoch 2
-------------------------------
loss: 0.268153  [   64/60000]
loss: 0.357762  [ 6464/60000]
loss: 0.310204  [12864/60000]
loss: 0.389778  [19264/60000]
loss: 0.403661  [25664/60000]
loss: 0.369263  [32064/60000]
loss: 0.307475  [38464/60000]
loss: 0.481929  [44864/60000]
loss: 0.393404  [51264/60000]
loss: 0.487791  [57664/60000]
Test Error: 
 Accuracy: 86.2%, Avg loss: 0.376983 

Epoch 3
-------------------------------
loss: 0.202981  [   64/60000]
loss: 0.331045  [ 6464/60000]
loss: 0.231186  [12864/60000]
loss: 0.307697  [19264/60000]
loss: 0.365355  [25664/60000]
loss: 0.352430  [32064/600

In [30]:
print()